In [1]:
from ml_utils.src import *
import joblib
import pandas as pd
from scipy.stats import norm
import random

In [2]:
def pipeline_aluno(dt_presenca, dt_desempenho, dados_aluno):
    
    df = pd.DataFrame([dados_aluno])
    X  = pre_processor_inferencia(df)

    # Probabilidades de presença
    prob_presenca = dt_presenca.predict_proba(X)[0]
    presenca      = dt_presenca.predict(X)[0]

    resultado = {
        'presenca':         'presente' if presenca == 1 else 'ausente',
        'prob_presente':    f'{prob_presenca[1]:.1%}',
        'prob_ausente':     f'{prob_presenca[0]:.1%}',
        'desempenho':       None,
        'prob_acima':       None,
        'prob_abaixo':      None,
    }

    if presenca == 1:
        prob_desempenho = dt_desempenho.predict_proba(X)[0]
        desempenho      = dt_desempenho.predict(X)[0]

        resultado['desempenho']  = 'acima da mediana' if desempenho == 1 else 'abaixo da mediana'
        resultado['prob_acima']  = f'{prob_desempenho[1]:.1%}'
        resultado['prob_abaixo'] = f'{prob_desempenho[0]:.1%}'

    return resultado

In [3]:
def chances_por_curso(prob_acima: float, desvio=0.10):
    cursos = {
    'Medicina (top)'        : 0.97, 
    'Direito (top)'         : 0.90,
    'Computação (top)'      : 0.85,
    'Engenharia'            : 0.74,
    'Administração'         : 0.65,
    'Pedagogia'             : 0.54,
    'Licenciaturas'         : 0.53,
    'Tecnólogos'            : 0.49,
    'Cursos noturnos'       : 0.42,
}
    resultado = {}
    for curso, percentil_corte in cursos.items():
        # Probabilidade do aluno superar o corte dado a incerteza
        z = (prob_acima - percentil_corte) / desvio
        chance = norm.cdf(z)
        resultado[curso] = f'{chance:.1%}'
        
    return resultado

In [7]:
def gerar_aluno_aleatorio():
    return {
        'Q001': random.choice(list('ABCDEFGH')),
        'Q002': random.choice(list('ABCDEFGH')),
        'Q003': random.choice(list('ABCDEF')),
        'Q004': random.choice(list('ABCDEF')),
        'Q005': random.randint(1,20 ),
        'Q006': random.choice(list('ABCDEFGHIJKLMNOPQ')),
        'Q007': random.choice(list('ABCD')),
        'Q008': random.choice(list('ABCDE')),
        'Q009': random.choice(list('ABCDE')),
        'Q010': random.choice(list('ABCDE')),
        'Q011': random.choice(list('ABCDE')),
        'Q012': random.choice(list('ABCDE')),
        'Q013': random.choice(list('ABCDE')),
        'Q014': random.choice(list('ABCDE')),
        'Q015': random.choice(list('ABCDE')),
        'Q016': random.choice(list('ABCDE')),
        'Q017': random.choice(list('ABCDE')),
        'Q018': random.choice(list('ABCDE')),
        'Q019': random.choice(list('ABCDE')),
        'Q020': random.choice(list('ABCDE')),
        'Q021': random.choice(list('ABCDE')),
        'Q022': random.choice(list('ABCDE')),
        'Q023': random.choice(list('ABCDE')),
        'Q024': random.choice(list('ABCDE')),
        'Q025': random.choice(list('AB')),
        'TP_FAIXA_ETARIA':        random.randint(1, 10),
        'TP_ESTADO_CIVIL':        random.randint(1, 4),
        'TP_ESCOLA':              random.choice([2, 3]),
        'TP_ST_CONCLUSAO':        random.randint(1, 4),
        'IN_TREINEIRO':           random.choice([0, 1]),
        'TP_DEPENDENCIA_ADM_ESC': random.randint(1, 4),
        'TP_LOCALIZACAO_ESC':     random.choice([1, 2]),
        'TP_SIT_FUNC_ESC':        random.randint(1, 3),
    }

In [9]:
rf_presenca   = joblib.load('models/rf_presenca.joblib')
rf_desempenho = joblib.load('models/rf_desempenho.joblib')

FileNotFoundError: [Errno 2] No such file or directory: 'models/rf_presenca.joblib'

In [10]:
resultados = []

for i in range(10):
    aluno = gerar_aluno_aleatorio()
    resultado = pipeline_aluno(rf_presenca, rf_desempenho, aluno)
    resultados.append(resultado)
    print(f'Aluno {i+1}: {resultado}')

Aluno 1: {'presenca': 'ausente', 'prob_presente': '48.5%', 'prob_ausente': '51.5%', 'desempenho': None, 'prob_acima': None, 'prob_abaixo': None}
Aluno 2: {'presenca': 'ausente', 'prob_presente': '35.2%', 'prob_ausente': '64.8%', 'desempenho': None, 'prob_acima': None, 'prob_abaixo': None}
Aluno 3: {'presenca': 'ausente', 'prob_presente': '39.0%', 'prob_ausente': '61.0%', 'desempenho': None, 'prob_acima': None, 'prob_abaixo': None}
Aluno 4: {'presenca': 'ausente', 'prob_presente': '40.5%', 'prob_ausente': '59.5%', 'desempenho': None, 'prob_acima': None, 'prob_abaixo': None}
Aluno 5: {'presenca': 'ausente', 'prob_presente': '15.0%', 'prob_ausente': '85.0%', 'desempenho': None, 'prob_acima': None, 'prob_abaixo': None}
Aluno 6: {'presenca': 'ausente', 'prob_presente': '23.1%', 'prob_ausente': '76.9%', 'desempenho': None, 'prob_acima': None, 'prob_abaixo': None}


ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- NU_ANO


In [ ]:
alunos_acima = [r for r in resultados if r['desempenho'] != None]
print(f'\nAlunos com desempenho acima da mediana: {len(alunos_acima)}')


Alunos com desempenho acima da mediana: 7


In [ ]:
alunos_acima

[{'presenca': 'presente',
  'prob_presente': '57.5%',
  'prob_ausente': '42.5%',
  'desempenho': 'acima da mediana',
  'prob_acima': '57.2%',
  'prob_abaixo': '42.8%'},
 {'presenca': 'presente',
  'prob_presente': '61.6%',
  'prob_ausente': '38.4%',
  'desempenho': 'abaixo da mediana',
  'prob_acima': '49.5%',
  'prob_abaixo': '50.5%'},
 {'presenca': 'presente',
  'prob_presente': '63.9%',
  'prob_ausente': '36.1%',
  'desempenho': 'acima da mediana',
  'prob_acima': '53.9%',
  'prob_abaixo': '46.1%'},
 {'presenca': 'presente',
  'prob_presente': '70.8%',
  'prob_ausente': '29.2%',
  'desempenho': 'abaixo da mediana',
  'prob_acima': '45.3%',
  'prob_abaixo': '54.7%'},
 {'presenca': 'presente',
  'prob_presente': '60.3%',
  'prob_ausente': '39.7%',
  'desempenho': 'acima da mediana',
  'prob_acima': '52.9%',
  'prob_abaixo': '47.1%'},
 {'presenca': 'presente',
  'prob_presente': '71.3%',
  'prob_ausente': '28.7%',
  'desempenho': 'abaixo da mediana',
  'prob_acima': '39.2%',
  'prob_ab

In [ ]:
for i, aluno in enumerate(alunos_acima):
    prob = float(aluno['prob_acima'].strip('%')) / 100
    
    print(f'\n===== Aluno {i+1} =====')
    print(f"Presença: {aluno['presenca']} ({aluno['prob_presente']})")
    print(f"Desempenho: {aluno['desempenho']} | Prob. acima: {aluno['prob_acima']}")
    print(f"\nChances por curso:")
    
    for curso, chance in chances_por_curso(prob).items():
        print(f'  {curso:<25} {chance}')



===== Aluno 1 =====
Presença: presente (57.5%)
Desempenho: acima da mediana | Prob. acima: 57.2%

Chances por curso:
  Medicina (top)            0.0%
  Direito (top)             0.1%
  Computação (top)          0.3%
  Engenharia                4.6%
  Administração             21.8%
  Pedagogia                 62.6%
  Licenciaturas             66.3%
  Tecnólogos                79.4%
  Cursos noturnos           93.6%

===== Aluno 2 =====
Presença: presente (61.6%)
Desempenho: abaixo da mediana | Prob. acima: 49.5%

Chances por curso:
  Medicina (top)            0.0%
  Direito (top)             0.0%
  Computação (top)          0.0%
  Engenharia                0.7%
  Administração             6.1%
  Pedagogia                 32.6%
  Licenciaturas             36.3%
  Tecnólogos                52.0%
  Cursos noturnos           77.3%

===== Aluno 3 =====
Presença: presente (63.9%)
Desempenho: acima da mediana | Prob. acima: 53.9%

Chances por curso:
  Medicina (top)            0.0%
  Direito